# Signal Modulation & Transmission

Transmitting a low-frequency signal (voice, data, music) directly through air is impractical:

- An antenna must be ~λ/4 long → at 1 kHz that means **75 km** of antenna
- Every signal would occupy the same frequency band — no channel separation
- Attenuation at low frequencies is severe

**Modulation** maps the *message* $m(t)$ onto a high-frequency **carrier**:

$$c(t) = A_c \cos(2\pi f_c t)$$

The carrier shuttles the message through the channel; its frequency $f_c$ determines
which band the signal occupies (radio stations, Wi-Fi channels, etc.). Three carrier
parameters can be varied — **amplitude**, **frequency**, and **phase** — giving rise to
AM, FM, and PM respectively.

The ratio $f_c / f_m \gg 1$ is essential: the carrier must oscillate many times per
message cycle so it can faithfully "carry" the message shape.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

%matplotlib inline
plt.rcParams.update({'figure.dpi': 100, 'font.size': 10})

def show_carrier(fc=80, fm=5):
    t = np.linspace(0, 0.6, 6000)
    m = np.cos(2 * np.pi * fm * t)
    c = np.cos(2 * np.pi * fc * t)
    s = (1 + 0.8 * m) * c          # AM preview

    clear_output(wait=True)
    fig, axes = plt.subplots(3, 1, figsize=(11, 6), sharex=True)

    axes[0].plot(t, m, color='steelblue', lw=1.8)
    axes[0].set(ylabel='m(t)', ylim=(-1.6, 1.6),
                title=f'Message  fm = {fm} Hz   [baseband]')
    axes[0].axhline(0, color='k', lw=0.4)

    axes[1].plot(t, c, color='darkorange', lw=0.7)
    axes[1].set(ylabel='c(t)', ylim=(-1.6, 1.6),
                title=f'Carrier  fc = {fc} Hz   [fc/fm = {fc//max(fm,1)}x faster]')

    axes[2].plot(t, s, color='seagreen', lw=0.7, label='modulated s(t)')
    axes[2].plot(t,  1 + 0.8*m, 'k--', lw=1.3, label='envelope')
    axes[2].plot(t, -(1 + 0.8*m), 'k--', lw=1.3)
    axes[2].set(ylabel='s(t)', ylim=(-2.2, 2.2), xlabel='Time (s)',
                title='Message imprinted on carrier — the envelope traces m(t)')
    axes[2].legend(loc='upper right', fontsize=9)

    plt.tight_layout()
    plt.show()

style = {'description_width': 'initial'}
widgets.interact(
    show_carrier,
    fc=widgets.IntSlider(value=80, min=20, max=200, step=5,
                         description='Carrier fc (Hz):', style=style),
    fm=widgets.IntSlider(value=5, min=1, max=20, step=1,
                         description='Message fm (Hz):', style=style)
)

interactive(children=(IntSlider(value=80, description='Carrier fc (Hz):', max=200, min=20, step=5, style=Slide…

<function __main__.show_carrier(fc=80, fm=5)>

## Amplitude Modulation (AM)

The message scales the carrier amplitude:

$$\boxed{s_{AM}(t) = A_c \bigl[1 + \mu\, m_n(t)\bigr] \cos(2\pi f_c t)}$$

$m_n(t)$ is the message normalised to unit peak; $\mu \in [0,1]$ is the **modulation index** (depth).

**Envelope** $A_c[1+\mu m_n(t)]$ is the outline shape — a diode + RC circuit recovers it (envelope detector). For $\mu > 1$ the envelope crosses zero: *overmodulation* → distortion.

**Spectrum** — multiplying by a cosine frequency-shifts the baseband spectrum by $\pm f_c$:

$$S_{AM}(f) = \underbrace{\tfrac{A_c}{2}\delta(f \pm f_c)}_{\text{carrier}} + \underbrace{\tfrac{\mu A_c}{4}\left[\delta(f\pm(f_c-f_m)) + \delta(f\pm(f_c+f_m))\right]}_{\text{two sidebands}}$$

Bandwidth $= 2f_m$. The carrier spike carries **no information** — it wastes transmit power. **DSB-SC** (Double Sideband Suppressed Carrier) eliminates it:

$$s_{DSB\text{-}SC}(t) = A_c \, m(t) \cos(2\pi f_c t)$$

Same bandwidth, twice the efficiency, but demodulation requires a phase-synchronised local carrier.

In [2]:
def plot_am(fc=100, fm=8, mu=0.8, dsb_sc=False):
    t  = np.linspace(0, 0.6, 8000)
    dt = t[1] - t[0]
    mn = np.cos(2 * np.pi * fm * t)
    cr = np.cos(2 * np.pi * fc * t)

    if dsb_sc:
        s   = mn * cr
        env = np.abs(mn)
        ttl = 'DSB-SC   s(t) = m(t) cos(2pi fc t)   [no carrier spike]'
        ec  = 'purple'
    else:
        s   = (1 + mu * mn) * cr
        env = 1 + mu * mn
        tag = '  OVERMODULATED' if mu > 1 else ''
        ttl = f'AM   mu = {mu:.2f}{tag}'
        ec  = 'red'

    N    = len(t)
    freq = np.fft.rfftfreq(N, dt)
    sp   = np.abs(np.fft.rfft(s)) / N

    clear_output(wait=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    mask = t < 0.4
    ax1.plot(t[mask], s[mask],    color='steelblue', lw=0.8, label='s(t)')
    ax1.plot(t[mask], env[mask],  color=ec, lw=1.6, ls='--', label='envelope')
    ax1.plot(t[mask], -env[mask], color=ec, lw=1.6, ls='--')
    ax1.set(xlabel='Time (s)', ylim=(-2.3, 2.3), title=ttl)
    ax1.legend(fontsize=9)
    ax1.axhline(0, color='k', lw=0.4)

    f_lo = max(0, fc - 4*fm - 5)
    f_hi = fc + 4*fm + 5
    fmask = (freq >= f_lo) & (freq <= f_hi)
    ax2.plot(freq[fmask], sp[fmask], color='darkorange', lw=1.5)
    ax2.axvline(fc,      color='gray',  ls=':', lw=1.0, label=f'fc = {fc} Hz')
    ax2.axvline(fc - fm, color='green', ls=':', lw=1.2, label=f'sidebands  fc +/- {fm} Hz')
    ax2.axvline(fc + fm, color='green', ls=':', lw=1.2)
    ax2.set(xlabel='Frequency (Hz)', title='Spectrum  |S(f)|')
    ax2.legend(fontsize=9)

    plt.suptitle(f'fc = {fc} Hz   fm = {fm} Hz   BW = {2*fm} Hz', fontsize=10)
    plt.tight_layout()
    plt.show()

style = {'description_width': 'initial'}
widgets.interact(
    plot_am,
    fc    =widgets.IntSlider(value=100, min=50,  max=300, step=10,   description='fc (Hz):', style=style),
    fm    =widgets.IntSlider(value=8,   min=1,   max=30,  step=1,    description='fm (Hz):', style=style),
    mu    =widgets.FloatSlider(value=0.8, min=0, max=1.5, step=0.05, description='mu (mod. index):', style=style),
    dsb_sc=widgets.Checkbox(value=False, description='DSB-SC mode (suppress carrier)', style=style)
)

interactive(children=(IntSlider(value=100, description='fc (Hz):', max=300, min=50, step=10, style=SliderStyle…

<function __main__.plot_am(fc=100, fm=8, mu=0.8, dsb_sc=False)>

## Frequency Modulation (FM)

The message controls the **instantaneous frequency** of the carrier:

$$f_i(t) = f_c + \Delta f \cdot m_n(t)$$

Integrating $f_i$ gives the instantaneous phase, so the transmitted signal is:

$$\boxed{s_{FM}(t) = A_c \cos\!\left(2\pi f_c t + 2\pi \Delta f \int_0^t m_n(\tau)\,d\tau\right)}$$

- $\Delta f$ — **frequency deviation** [Hz]: the maximum swing of $f_i$ around $f_c$
- $\beta = \Delta f / f_m$ — **FM modulation index**

**Carson's rule** — covers $\approx 98\%$ of the signal power:

$$B_{FM} \approx 2(\Delta f + f_m) = 2 f_m(1 + \beta)$$

For $\beta \ll 1$ (narrowband FM) the spectrum resembles AM with two sidebands.
For $\beta \gg 1$ (wideband FM) Bessel-function sidebands spread across $B_{FM}$.

**Noise immunity**: FM receivers *clip* amplitude variations before demodulation — amplitude noise is simply discarded. This gives FM a fundamental SNR advantage over AM; commercial FM radio trades bandwidth for audio quality.

In [3]:
def plot_fm(fc=100, fm=8, delta_f=20):
    t  = np.linspace(0, 0.6, 12000)
    dt = t[1] - t[0]
    mn = np.cos(2 * np.pi * fm * t)

    f_inst = fc + delta_f * mn
    phase  = 2*np.pi*fc*t + 2*np.pi*delta_f * np.cumsum(mn) * dt
    s      = np.cos(phase)

    beta      = delta_f / fm
    bw_carson = 2 * (delta_f + fm)

    N    = len(t)
    freq = np.fft.rfftfreq(N, dt)
    sp   = np.abs(np.fft.rfft(s)) / N

    clear_output(wait=True)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    mask = t < 0.4
    axes[0].plot(t[mask], s[mask], color='seagreen', lw=0.7)
    axes[0].set(xlabel='Time (s)', ylim=(-1.5, 1.5),
                title=f'FM signal   beta = Df/fm = {beta:.2f}')
    axes[0].axhline(0, color='k', lw=0.4)

    axes[1].plot(t[mask], f_inst[mask], color='purple', lw=1.5)
    axes[1].fill_between(t[mask], fc-delta_f, fc+delta_f, alpha=0.10, color='purple')
    axes[1].axhline(fc,           color='gray',   ls='--', lw=1.0, label=f'fc = {fc} Hz')
    axes[1].axhline(fc + delta_f, color='purple', ls=':', lw=1.2, label=f'+/-Df = {delta_f} Hz')
    axes[1].axhline(fc - delta_f, color='purple', ls=':', lw=1.2)
    axes[1].set(xlabel='Time (s)', ylabel='Instantaneous frequency (Hz)',
                title='fi(t) swings synchronously with message')
    axes[1].legend(fontsize=9)

    f_lo = max(0, fc - bw_carson * 0.7)
    f_hi = fc + bw_carson * 0.7
    fmask = (freq >= f_lo) & (freq <= f_hi)
    axes[2].plot(freq[fmask], sp[fmask], color='seagreen', lw=1.5)
    axes[2].axvline(fc - bw_carson/2, color='red', ls=':', lw=1.3,
                    label=f'Carson BW = {bw_carson:.0f} Hz')
    axes[2].axvline(fc + bw_carson/2, color='red', ls=':', lw=1.3)
    axes[2].set(xlabel='Frequency (Hz)', title='Spectrum  |S(f)|')
    axes[2].legend(fontsize=9)

    plt.suptitle(f'fc = {fc} Hz   fm = {fm} Hz   Df = {delta_f} Hz   beta = {beta:.2f}', fontsize=10)
    plt.tight_layout()
    plt.show()

style = {'description_width': 'initial'}
widgets.interact(
    plot_fm,
    fc     =widgets.IntSlider(value=100, min=50, max=300, step=10, description='fc (Hz):', style=style),
    fm     =widgets.IntSlider(value=8,  min=1,  max=30,  step=1,  description='fm (Hz):', style=style),
    delta_f=widgets.IntSlider(value=20, min=1,  max=100, step=1,  description='Df deviation (Hz):', style=style)
)

interactive(children=(IntSlider(value=100, description='fc (Hz):', max=300, min=50, step=10, style=SliderStyle…

<function __main__.plot_fm(fc=100, fm=8, delta_f=20)>

## Phase Modulation (PM) & Digital Modulations

PM shifts the carrier phase proportionally to the message:

$$\boxed{s_{PM}(t) = A_c \cos\!\bigl(2\pi f_c t + k_p\, m(t)\bigr)}$$

$k_p$ [rad/V] is the **phase sensitivity**; the peak phase deviation is $\Delta\phi = k_p A_m$.

### PM vs FM — the key distinction

The instantaneous frequency of the PM signal is:

$$f_i^{PM}(t) = f_c + \frac{k_p}{2\pi}\,\dot{m}(t) = f_c - k_p f_m A_m \sin(2\pi f_m t)$$

| | FM | PM |
|---|---|---|
| $f_i(t)$ follows | $m(t)$ directly | $\dot m(t)$ — the derivative |
| Peak freq. deviation | $\Delta f$ — **constant** | $k_p f_m$ — **grows with $f_m$** |
| Phase expression | $\propto \int m\,d\tau$ | $\propto m(t)$ |

Increasing $f_m$ **widens PM's frequency swing** but leaves FM's unchanged. The interactive below makes this visible.

### PM in discrete form: digital modulations

| Scheme | Phase alphabet | Bits/symbol |
|---|---|---|
| BPSK | $\{0,\, \pi\}$ | 1 |
| QPSK | $\{0,\,\tfrac{\pi}{2},\,\pi,\,\tfrac{3\pi}{2}\}$ | 2 |
| 8-PSK | 8 equally spaced phases | 3 |

BPSK, QPSK, and QAM are the backbone of Wi-Fi, 4G/5G, and GPS.

In [4]:
def plot_pm(fc=100, fm=8, kp=2.5, bpsk=False):
    t  = np.linspace(0, 0.6, 12000)
    dt = t[1] - t[0]
    mn = np.cos(2 * np.pi * fm * t)

    if bpsk:
        bits  = np.sign(mn)
        phase_shift = np.pi * (bits < 0).astype(float)
        s     = np.cos(2*np.pi*fc*t + phase_shift)
        color = 'slateblue'
        label = 'BPSK  phase in {0, pi}'
        f_inst_pm = fc + np.gradient(phase_shift, dt) / (2*np.pi)
    else:
        s     = np.cos(2*np.pi*fc*t + kp*mn)
        color = 'crimson'
        label = f'PM   peak phase dev = {kp:.2f} rad'
        dm_dt = -2*np.pi*fm * np.sin(2*np.pi*fm*t)
        f_inst_pm = fc + kp * dm_dt / (2*np.pi)

    # FM reference with fixed Df=20 Hz — shows FM swing is constant regardless of fm
    df_ref    = 20
    f_inst_fm = fc + df_ref * mn

    N    = len(t)
    freq = np.fft.rfftfreq(N, dt)
    sp   = np.abs(np.fft.rfft(s)) / N

    clear_output(wait=True)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    mask = t < 0.4
    axes[0].plot(t[mask], s[mask], color=color, lw=0.7)
    if not bpsk:
        axes[0].plot(t[mask], 0.4*mn[mask], 'k--', lw=1, alpha=0.55, label='m(t) scaled')
        axes[0].legend(fontsize=9)
    axes[0].set(xlabel='Time (s)', ylim=(-1.5, 1.5), title=label)
    axes[0].axhline(0, color='k', lw=0.4)

    pm_peak_df = kp * fm if not bpsk else None
    axes[1].plot(t[mask], f_inst_pm[mask], color=color, lw=1.5,
                 label=f'PM  peak Df = {pm_peak_df:.1f} Hz (= kp*fm)' if not bpsk else 'BPSK fi(t)')
    if not bpsk:
        axes[1].plot(t[mask], f_inst_fm[mask], color='seagreen', lw=1.2, ls='--',
                     label=f'FM ref  Df = {df_ref} Hz (fixed)')
        axes[1].fill_between(t[mask], fc-pm_peak_df, fc+pm_peak_df, alpha=0.08, color=color)
        axes[1].fill_between(t[mask], fc-df_ref, fc+df_ref, alpha=0.08, color='seagreen')
    axes[1].axhline(fc, color='gray', ls='--', lw=1, label=f'fc = {fc} Hz')
    axes[1].set(xlabel='Time (s)', ylabel='Hz',
                title='fi(t): PM swing grows with fm — FM swing stays constant')
    axes[1].legend(fontsize=8, loc='upper right')

    f_lo = max(0, fc * 0.5)
    f_hi = fc * 1.6
    fmask = (freq >= f_lo) & (freq <= f_hi)
    axes[2].plot(freq[fmask], sp[fmask], color=color, lw=1.5)
    axes[2].axvline(fc, color='gray', ls=':', lw=1)
    axes[2].set(xlabel='Frequency (Hz)', title='Spectrum  |S(f)|')

    plt.suptitle(f'fc = {fc} Hz   fm = {fm} Hz', fontsize=10)
    plt.tight_layout()
    plt.show()

style = {'description_width': 'initial'}
widgets.interact(
    plot_pm,
    fc  =widgets.IntSlider(value=100, min=50,  max=300, step=10,  description='fc (Hz):', style=style),
    fm  =widgets.IntSlider(value=8,   min=1,   max=30,  step=1,   description='fm (Hz):', style=style),
    kp  =widgets.FloatSlider(value=2.5, min=0.1, max=6, step=0.1, description='kp (rad/V):', style=style),
    bpsk=widgets.Checkbox(value=False, description='Switch to BPSK', style=style)
)

interactive(children=(IntSlider(value=100, description='fc (Hz):', max=300, min=50, step=10, style=SliderStyle…

<function __main__.plot_pm(fc=100, fm=8, kp=2.5, bpsk=False)>

## AM / FM / PM — Unified Comparison

| Property | AM | FM | PM |
|---|---|---|---|
| Modified parameter | Amplitude | Frequency | Phase |
| Bandwidth | $2f_m$ | $2(\Delta f + f_m)$ | $\approx 2(k_p A_m + 1)f_m$ |
| Noise immunity | Low | High | High |
| Demodulation | Envelope detector | Frequency discriminator | Phase-locked loop |
| Applications | AM radio, aviation | FM radio, TV audio | GPS, Wi-Fi, 4G/5G |

**Demodulation intuition:**

- **AM**: rectify → low-pass → the envelope *is* the message, readable with a simple diode.
- **FM**: differentiate the phase (limiter + discriminator) → $f_i - f_c \propto m(t)$.
- **PM**: track phase with a PLL → the phase error is proportional to $m(t)$.

Angle modulation (FM, PM) sacrifices bandwidth for noise robustness — as $\beta$ grows, spectral energy spreads but the output SNR *improves* (FM improvement theorem). This is why music broadcasting moved from AM to FM.

In [5]:
def compare_all(fc=100, fm=8, mu=0.7, delta_f=20, kp=2.5):
    t  = np.linspace(0, 0.8, 16000)
    dt = t[1] - t[0]
    mn = np.cos(2 * np.pi * fm * t)

    am   = (1 + mu * mn) * np.cos(2*np.pi*fc*t)
    ph   = 2*np.pi*fc*t + 2*np.pi*delta_f * np.cumsum(mn) * dt
    fm_s = np.cos(ph)
    pm_s = np.cos(2*np.pi*fc*t + kp*mn)

    N    = len(t)
    freq = np.fft.rfftfreq(N, dt)
    def sp(sig): return np.abs(np.fft.rfft(sig)) / N

    bws     = [2*fm, 2*(delta_f+fm), 2*(kp*fm+fm)]
    signals = [am, fm_s, pm_s]
    spectra = [sp(s) for s in signals]
    names   = ['AM', 'FM', 'PM']
    colors  = ['steelblue', 'seagreen', 'crimson']
    params  = [f'mu={mu:.2f}', f'Df={delta_f} Hz  beta={delta_f/fm:.1f}', f'kp={kp:.1f} rad/V']

    max_bw = max(bws)
    f_lo   = max(0, fc - max_bw * 0.65)
    f_hi   = fc + max_bw * 0.65
    fmask  = (freq >= f_lo) & (freq <= f_hi)
    tmask  = t < 0.4

    clear_output(wait=True)
    fig, axes = plt.subplots(3, 2, figsize=(14, 9))

    for i, (sig, sp_i, nm, col, par, bw_i) in enumerate(
            zip(signals, spectra, names, colors, params, bws)):

        axes[i,0].plot(t[tmask], sig[tmask], color=col, lw=0.7)
        axes[i,0].set(ylim=(-2.1, 2.1))
        axes[i,0].set_title(f'{nm}  [{par}]', color=col, fontweight='bold')
        axes[i,0].axhline(0, color='k', lw=0.4)
        if i == 2: axes[i,0].set_xlabel('Time (s)')

        axes[i,1].plot(freq[fmask], sp_i[fmask], color=col, lw=1.5)
        axes[i,1].axvline(fc, color='gray', ls=':', lw=0.8)
        axes[i,1].axvline(fc - bw_i/2, color=col, ls='--', lw=1.1, alpha=0.6,
                          label=f'BW = {bw_i:.0f} Hz')
        axes[i,1].axvline(fc + bw_i/2, color=col, ls='--', lw=1.1, alpha=0.6)
        axes[i,1].legend(fontsize=9, loc='upper right')
        if i == 2: axes[i,1].set_xlabel('Frequency (Hz)')

    axes[0,0].set_title('Time domain', fontsize=11)
    axes[0,1].set_title('Spectrum  |S(f)|', fontsize=11)

    plt.suptitle(f'fc = {fc} Hz   fm = {fm} Hz', fontsize=11, y=1.005)
    plt.tight_layout()
    plt.show()

style = {'description_width': 'initial'}
widgets.interact(
    compare_all,
    fc     =widgets.IntSlider(value=100, min=50,  max=300, step=10,   description='fc (Hz):', style=style),
    fm     =widgets.IntSlider(value=8,   min=1,   max=30,  step=1,    description='fm (Hz):', style=style),
    mu     =widgets.FloatSlider(value=0.7, min=0, max=1.0, step=0.05, description='AM mu:', style=style),
    delta_f=widgets.IntSlider(value=20,  min=1,  max=80,  step=1,    description='FM Df (Hz):', style=style),
    kp     =widgets.FloatSlider(value=2.5, min=0.1, max=6, step=0.1, description='PM kp (rad/V):', style=style)
)

interactive(children=(IntSlider(value=100, description='fc (Hz):', max=300, min=50, step=10, style=SliderStyle…

<function __main__.compare_all(fc=100, fm=8, mu=0.7, delta_f=20, kp=2.5)>